In [1]:
import pandas as pd

parquet_sample = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\data\01-12\packet-level\packets\SAT-01-12-2018_090.parquet"

df = pd.read_parquet(parquet_sample)

In [5]:
df.shape

(404023, 25)

In [2]:
df.columns

Index(['timestamp', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol',
       'length', 'ttl', 'payload_size', 'ip_id', 'ip_flags',
       'ip_fragment_offset', 'ip_tos', 'tcp_flags', 'tcp_window', 'tcp_seq',
       'tcp_ack', 'tcp_urgent', 'icmp_type', 'icmp_code', 'flow_id',
       'packet_position', 'inter_arrival_time', 'payload_entropy',
       'packet_direction'],
      dtype='object')

In [6]:
df.head(5)

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,length,ttl,payload_size,ip_id,...,tcp_seq,tcp_ack,tcp_urgent,icmp_type,icmp_code,flow_id,packet_position,inter_arrival_time,payload_entropy,packet_direction
0,1.543675e+09,172.16.0.5,192.168.50.1,876.0,26148.0,17,482,50,440,0,...,NaN,NaN,NaN,NaN,NaN,172.16.0.5_192.168.50.1_876_26148_17,1,0.000000e+00,3.857682,forward
1,1.543675e+09,172.16.0.5,192.168.50.1,634.0,5731.0,17,482,48,440,0,...,NaN,NaN,NaN,NaN,NaN,172.16.0.5_192.168.50.1_634_5731_17,1,0.000000e+00,3.570195,forward
2,1.543675e+09,172.16.0.5,192.168.50.1,876.0,26148.0,17,482,50,440,0,...,NaN,NaN,NaN,NaN,NaN,172.16.0.5_192.168.50.1_876_26148_17,2,1.907349e-06,3.696960,forward
3,1.543675e+09,172.16.0.5,192.168.50.1,876.0,26148.0,17,482,50,440,0,...,NaN,NaN,NaN,NaN,NaN,172.16.0.5_192.168.50.1_876_26148_17,3,0.000000e+00,3.857682,forward
4,1.543675e+09,172.16.0.5,192.168.50.1,876.0,26148.0,17,482,50,440,0,...,NaN,NaN,NaN,NaN,NaN,172.16.0.5_192.168.50.1_876_26148_17,4,9.536743e-07,3.696960,forward


In [1]:
import os
import pandas as pd
from collections import defaultdict
import pyarrow.parquet as pq
import pickle

# Folder containing all packet-level parquet files
packet_folder = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\data\01-12\packet-level\packets"

# Dictionary to store mapping: file_name -> set of flow_ids in that file
file_to_flow_ids = defaultdict(set)

# List all packet files
packet_files = os.listdir(packet_folder)

# Process each packet file
for file_name in packet_files:
    file_path = os.path.join(packet_folder, file_name)

    # Open parquet file
    parquet_file = pq.ParquetFile(file_path)
    num_rows = parquet_file.metadata.num_rows
    batch_size = 20000

    # Read the entire file at once (more efficient for this use case)
    # If memory is a concern, you can keep the batch approach with proper row slicing
    batch_df = parquet_file.read().to_pandas()

    # Collect flow_ids from this file
    flow_ids_in_file = set(batch_df['flow_id'].unique())
    file_to_flow_ids[file_name].update(flow_ids_in_file)

    print(f"Processed file: {file_name}, total unique flow_ids so far: {len(file_to_flow_ids[file_name])}")

# Save the mapping for later use
saving_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\mapping\packet_level_mapping\packet_file_flow_mapping.pkl"

# Create the directory structure if it doesn't exist
os.makedirs(os.path.dirname(saving_path), exist_ok=True)

with open(saving_path, "wb") as f:
    pickle.dump(file_to_flow_ids, f)

print("Done building flow_id → file mapping!")
print(f"Mapping saved to: {saving_path}")

Processed file: SAT-01-12-2018_0.parquet, total unique flow_ids so far: 16927
Processed file: SAT-01-12-2018_01.parquet, total unique flow_ids so far: 5351
Processed file: SAT-01-12-2018_010.parquet, total unique flow_ids so far: 5938
Processed file: SAT-01-12-2018_0100.parquet, total unique flow_ids so far: 6132
Processed file: SAT-01-12-2018_0101.parquet, total unique flow_ids so far: 6242
Processed file: SAT-01-12-2018_0102.parquet, total unique flow_ids so far: 6686
Processed file: SAT-01-12-2018_0103.parquet, total unique flow_ids so far: 6866
Processed file: SAT-01-12-2018_0104.parquet, total unique flow_ids so far: 7082
Processed file: SAT-01-12-2018_0105.parquet, total unique flow_ids so far: 6869
Processed file: SAT-01-12-2018_0106.parquet, total unique flow_ids so far: 6979
Processed file: SAT-01-12-2018_0107.parquet, total unique flow_ids so far: 7030
Processed file: SAT-01-12-2018_0108.parquet, total unique flow_ids so far: 7125
Processed file: SAT-01-12-2018_0109.parquet, 

In [2]:
import pickle

file_path = r"/models/model_main/data/mapping/packet_level_mapping/packet_file_flow_mapping_no_duplicates.pkl"

try:
    with open(file_path, "rb") as f:
        packet_file_flow_mapping = pickle.load(f)

    # Inspect type and show a small sample (first 5 items)
    mapping_type = type(packet_file_flow_mapping)
    sample_items = list(packet_file_flow_mapping.items())[:5]

    var = (mapping_type, sample_items)
    print(var)
except Exception as e:
    str(e)

(<class 'collections.defaultdict'>, [('SAT-01-12-2018_0.parquet', {'4.2.2.4_192.168.50.8_53_55582_17', '192.168.50.7_4.2.2.4_51532_53_17', '192.168.50.8_4.2.2.4_56752_53_17', '4.2.2.4_192.168.50.8_53_52458_17', '192.168.50.6_23.194.110.121_56489_443_6', '4.2.2.4_192.168.50.8_53_65332_17', '4.2.2.4_192.168.50.8_53_62536_17', '192.168.50.8_4.2.2.4_64539_53_17', '52.114.75.78_192.168.50.7_443_50790_6', '192.168.50.8_104.18.59.178_58736_80_6', '52.88.72.192_192.168.50.8_443_58950_6', '4.2.2.4_192.168.50.7_53_53940_17', '4.2.2.4_192.168.50.8_53_63440_17', '192.168.50.7_8.8.8.8_54402_53_17', '192.168.50.1_172.16.0.5_80_21696_6', '192.168.50.7_4.2.2.4_63051_53_17', '192.168.50.6_172.217.9.226_56795_443_6', '172.16.0.5_192.168.50.1_60778_80_6', '4.2.2.4_192.168.50.8_53_55332_17', '192.168.50.6_4.2.2.4_54659_53_17', '4.2.2.4_192.168.50.6_53_59154_17', '4.2.2.4_192.168.50.6_53_51984_17', '192.168.50.6_4.2.2.4_50368_53_17', '23.52.155.27_192.168.50.7_80_50551_6', '192.168.50.7_35.173.44.140_50620

In [3]:
import pickle
from collections import Counter

file_path = r"/models/model_main/data/mapping/packet_level_mapping/packet_file_flow_mapping_no_duplicates.pkl"

with open(file_path, "rb") as f:
    packet_file_flow_mapping = pickle.load(f)

all_flows = []
for file, flows in packet_file_flow_mapping.items():
    all_flows.extend(flows)

total_flows = len(all_flows)

unique_flows = len(set(all_flows))

duplicates_count = total_flows - unique_flows
duplicates = [item for item, count in Counter(all_flows).items() if count > 1]

print(f"Total flows: {total_flows}")
print(f"Unique flows: {unique_flows}")
print(f"Number of duplicate flows: {duplicates_count}")
print(f"Example duplicate flow IDs: {duplicates[:10]}")

Total flows: 50339714
Unique flows: 28574738
Number of duplicate flows: 21764976
Example duplicate flow IDs: ['4.2.2.4_192.168.50.8_53_55582_17', '192.168.50.8_4.2.2.4_56752_53_17', '4.2.2.4_192.168.50.8_53_63440_17', '4.2.2.4_192.168.50.8_53_52563_17', '4.2.2.4_192.168.50.8_53_55447_17', '192.168.50.7_4.2.2.4_50818_53_17', '172.16.0.5_192.168.50.1_634_59722_17', '4.2.2.4_192.168.50.8_53_63930_17', '172.16.0.5_192.168.50.1_634_21303_17', '4.2.2.4_192.168.50.6_53_59251_17']


In [3]:
import pickle

file_path = r"/models/model_main/data/mapping/packet_level_mapping/packet_file_flow_mapping_no_duplicates.pkl"

with open(file_path, "rb") as f:
    mapping = pickle.load(f)

print(type(mapping))
if isinstance(mapping, list):
    print("First 2 elements:", mapping[:2])
elif isinstance(mapping, dict):
    print("Sample 2 items:", list(mapping.items())[:2])

<class 'collections.defaultdict'>
Sample 2 items: [('SAT-01-12-2018_0.parquet', {'192.168.50.8_4.2.2.4_63155_53_17', '4.2.2.4_192.168.50.6_53_51524_17', '172.217.3.98_192.168.50.8_443_58827_6', '8.250.131.254_192.168.50.7_80_50697_6', '172.16.0.5_192.168.50.1_634_39959_17', '4.2.2.4_192.168.50.7_53_65336_17', '192.168.50.7_4.2.2.4_51477_53_17', '192.168.50.1_172.16.0.5_80_51438_6', '192.168.50.8_172.217.12.134_58831_443_6', '192.168.50.8_4.2.2.4_61070_53_17', '4.2.2.4_192.168.50.8_53_62046_17', '8.8.8.8_192.168.50.8_53_56894_17', '172.16.0.5_192.168.50.1_36035_80_6', '4.2.2.4_192.168.50.6_53_63316_17', '4.2.2.4_192.168.50.7_53_57210_17', '34.232.238.166_192.168.50.8_443_59021_6', '192.168.50.8_74.208.236.171_58742_80_6', '4.2.2.4_192.168.50.6_53_51265_17', '192.168.50.8_8.8.8.8_57294_53_17', '4.2.2.4_192.168.50.8_53_56706_17', '4.2.2.4_192.168.50.7_53_54956_17', '192.168.50.8_4.2.2.4_54782_53_17', '192.168.50.6_173.241.244.143_56443_443_6', '192.168.50.8_8.8.8.8_56500_53_17', '4.2.2.4_

In [4]:
import pickle

file_path = r"/models/model_main/data/mapping/packet_level_mapping/packet_file_flow_mapping_no_duplicates.pkl"

# Load
with open(file_path, "rb") as f:
    file_to_flow_ids = pickle.load(f)

print("Original entries (file → set):", len(file_to_flow_ids))

# Invert mapping: flow_id → file_name (keep only one file if duplicates exist)
flow_to_file = {}

for file_name, flow_ids in file_to_flow_ids.items():
    for fid in flow_ids:
        if fid not in flow_to_file:   # keep first occurrence only
            flow_to_file[fid] = file_name

print("Unique flows after cleaning:", len(flow_to_file))

# Overwrite with cleaned mapping
with open(file_path, "wb") as f:
    pickle.dump(flow_to_file, f)

print("File overwritten: now it's flow_id → file_name (unique flows only)")

Original entries (file → set): 819
Unique flows after cleaning: 28574738
File overwritten: now it's flow_id → file_name (unique flows only)


In [5]:
import pickle

file_path = r"/models/model_main/data/mapping/packet_level_mapping/packet_file_flow_mapping_no_duplicates.pkl"

# Load the pickle file
with open(file_path, "rb") as f:
    mapping = pickle.load(f)

print(f"Total flows: {len(mapping)}")
print(f"Unique flows: {len(set(mapping.keys()))}")

if len(mapping) != len(set(mapping.keys())):
    print("Duplicate flow_ids found!")
else:
    print("All flow_ids are unique.")

Total flows: 28574738
Unique flows: 28574738
All flow_ids are unique.
